# Topology-Aware Physics-Informed GNN for AC Power Flow
### Reproduce all results and figures on Google Colab

This notebook regenerates **every** result and figure for the topology-aware physics-informed GNN (PIGNN) surrogate for AC power flow on the IEEE 33-bus distribution feeder (pandapower `case33bw`). The model is a `ProxySolverGNN` (3x TAGConv, default hop counts `ks=(3,3,2)`, with per-unit series-admittance edge weights) trained against Newton-Raphson (NR) solutions and optionally fine-tuned with a differentiable AC power-balance residual.

**Important note on where the wall-clock goes.** The GPU accelerates **training** (the TAGConv message passing). But the training/eval data is generated by running Newton-Raphson power flows with **pandapower**, which is **CPU-bound** and cannot use the GPU. So on Colab most of the wall-clock is spent in scenario/data generation, not in the GPU training loop. This is still much faster than a slow local machine (Colab gives you a fast multi-core CPU plus a GPU), but do not expect the GPU to shrink the data-generation phase.

**Before you run anything:** set **Runtime -> Change runtime type -> Hardware accelerator: GPU**, then run the cells top to bottom.

In [ ]:
# Check the GPU (training only; NR data generation stays on CPU)
!nvidia-smi

## Step 1 — get the code

**Option B (default): upload the project zip.** A clean zip named `topology-aware-pignn-colab.zip` (committed files only — no virtualenv or checkpoints) was prepared for you. Run the **Option B** cell below, choose that zip in the upload dialog, and it will unzip and enter the project directory.

**Option A: clone from GitHub.** Use this *instead* only once you have pushed the enhanced code to GitHub. If you use Option A, uncomment its cell and skip Option B.

In [ ]:
# Option A — clone from GitHub (use ONLY after you have pushed the enhanced code).
# Commented out by default; the working path below is Option B (upload the zip).
# !git clone https://github.com/ra-emami/topology-aware-gnn-power-flow.git
# %cd topology-aware-gnn-power-flow

In [ ]:
# Option B (default) — upload the clean project zip prepared for you.
# Run this cell, pick 'topology-aware-pignn-colab.zip' in the dialog; it unzips and enters the project.
from google.colab import files
import zipfile
uploaded = files.upload()            # choose topology-aware-pignn-colab.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall('.')
%cd topology-aware-pignn

In [ ]:
# Install the package (with test extras). numpy<2.1 and pandas==2.2.2 are pinned by the project.
!pip install -e ".[test]" -q

In [ ]:
# Run the test suite (5 tests: topology build structure; radial spanning-tree reconfigurations;
# Ybus power-balance residual ~0 on a true NR solution; LinDistFlow zero-injection flat; LinDistFlow tracks NR)
!python -m pytest -q

## Step 2 - full pipeline (this is the heavy part)

The commands below regenerate all checkpoints and result files. Most of the wall-clock here is CPU-bound Newton-Raphson scenario generation (pandapower), so this phase takes a while even with a GPU attached. Each script also accepts `--quick` for a fast smoke run; the arguments below reproduce the full reference results.

Reference outputs this produces (a fresh run may vary slightly with seed/hardware):

1. **Supervised** (5000 scenarios, 300 epochs): Voltage R2=0.984, Voltage MAE=2.55 mV/pu, Angle R2=0.931, Angle MAE=0.12 deg, power residual |dP|=0.260 MW.
2. **Physics-informed fine-tuning** (lambda=10), before -> after: |dP| 0.252 -> 0.125 MW (~50% lower), |dQ| 0.216 -> 0.118 MVAr (~46% lower) at negligible accuracy cost - roughly halving the AC power-balance violation.
3. **Topology-general** (6000 scenarios, 42 seen / 17 held-out reconfigurations): held-out MAE 17.8 mV/pu vs 20.4 for the single-topology model - training across reconfigurations transfers to unseen topologies.
4. **K / edge-weight ablation** (single base topology): hop count K dominates single-topology accuracy (K1=4.51 -> K4=1.98 with edge weights ON; best K=4 edge-weights-off MAE=1.77, R2=0.990).
5. **Edge-weight topology ablation** (multi-topology, 3 seeds): edge weights yield a smaller generalization gap in 3/3 seeds (mean 7.26 vs 8.47 mV/pu) and lower held-out MAE in 2/3 (mean 19.18 vs 19.76, ~2.9%, within noise) - the seed-consistent gap reduction supports "topology-aware".
6. **NR warm-start** (500 scenarios): NR iterations 3.26 -> 2.99 (8.2% fewer), never worse than flat start, identical solution (max |V| discrepancy ~7e-9 pu).

In [ ]:
# 1) Supervised training -> checkpoints/gnn_powerflow.pt + results/supervised.json
!python scripts/train.py --samples 5000 --epochs 300

# 2) Physics-informed fine-tuning -> checkpoints/gnn_powerflow_pinn.pt + results/physics_finetune.json
!python scripts/finetune_physics.py --ckpt checkpoints/gnn_powerflow.pt --lam 10

# 2b) Full evaluation report: baselines + physics consistency + topology generalization
!python scripts/evaluate.py --ckpt checkpoints/gnn_powerflow_pinn.pt

# 3) Topology-general training -> checkpoints/gnn_topogeneral.pt + results/topology_general.json
!python scripts/train_topology_general.py --samples 6000 --epochs 200

# 4) Hop-count K ablation -> results/ablation_k.csv
!python scripts/ablate_k.py --ks 1 2 3 4

# 5) Edge-weight x topology ablation -> results/ablation_edgeweights_topology.json
!python scripts/ablate_edgeweights_topology.py --samples 6000 --epochs 200 --seeds 42 43 44

# 6) NR warm-start -> results/warmstart_nr.json + results/warmstart_nr_per_scenario.csv
!python scripts/warmstart_nr.py --n 500

In [ ]:
# Render all figures from results/ + checkpoints -> figures/*.png
# (training_curve.png, parity.png, baseline_comparison.png, physics_before_after.png,
#  ablation_k.png, ablation_edgeweights_topology.png, warmstart_nr.png)
!python scripts/make_figures.py

## Step 3 - download the artifacts and send them back

Zip up the `results/` and `figures/` directories and download the bundle to your local machine.

In [ ]:
# Bundle results + figures and download
!zip -r artifacts.zip results figures
from google.colab import files
files.download('artifacts.zip')